# Exploration Article 9 RGPD

Ce notebook lit les exports produits par `python src/run_article9_scan.py` et donne quelques vues rapides pour explorer les resultats.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
repo_root = Path.cwd()
if not (repo_root / "data").exists() and (repo_root.parent / "data").exists():
    repo_root = repo_root.parent

repo_root

In [ ]:
# Decommente cette cellule pour relancer le scan depuis le notebook.
# import subprocess
# import sys
# subprocess.run([sys.executable, str(repo_root / "src" / "run_article9_scan.py")], check=True)

processed_dir = repo_root / "data" / "processed"
findings_path = processed_dir / "article9_findings.csv"
summary_path = processed_dir / "article9_summary.csv"

if not findings_path.exists() or not summary_path.exists():
    raise FileNotFoundError(
        "Les fichiers de sortie sont absents. Lance d'abord : python src/run_article9_scan.py"
    )

findings = pd.read_csv(findings_path)
summary = pd.read_csv(summary_path)

findings.head(), summary.head()

In [ ]:
summary.sort_values(["finding_count", "max_score"], ascending=[False, False])

In [ ]:
entity_counts = findings.groupby("entity_name").size().sort_values(ascending=False)
entity_counts

In [ ]:
ax = entity_counts.plot(kind="bar", figsize=(10, 4), color="#355C7D")
ax.set_title("Nombre de detections par entite")
ax.set_xlabel("Entite")
ax.set_ylabel("Detections")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

In [ ]:
file_counts = findings.groupby("file_name").size().sort_values(ascending=False)
ax = file_counts.plot(kind="bar", figsize=(12, 4), color="#C06C84")
ax.set_title("Nombre de detections par fichier")
ax.set_xlabel("Fichier")
ax.set_ylabel("Detections")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()

In [ ]:
pivot = (
    findings.pivot_table(
        index="file_name",
        columns="entity_name",
        values="matched_text",
        aggfunc="count",
        fill_value=0,
    )
    .sort_index()
)
pivot.style.background_gradient(cmap="OrRd")

In [ ]:
focus_file = summary.sort_values("finding_count", ascending=False).iloc[0]["file_name"]
focus = findings.loc[findings["file_name"] == focus_file].sort_values(["page_number", "adjusted_score"], ascending=[True, False])
focus[["file_name", "page_number", "entity_name", "matched_text", "adjusted_score", "context_hits", "risk_level"]]